# vla-reachability: rare unsafe-state reachability of VLA policies on LIBERO via multilevel splitting over action-chunk branches
**Branch:** `feature/vla-reachability` | **Gate:** frozen in README.md (2026-11-08)
**One-Click Execution:** Runtime -> Run all (GPU: L4). Clones, installs LeRobot 0.7 + LIBERO, runs pytest on the GPU machine
(nothing may be skipped; aborts if invalid), runs pilot + resumable campaign with results on Google Drive, prints RESULTS.md,
unassigns the GPU. Re-running after a disconnect resumes from Drive.

In [ ]:
# Step 1: Credit Guard (auto-unassign on any error or completion) + Colab Secrets
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print("\n[Credit Guard] Error, interrupt, or completion! Flushing Drive and unassigning GPU...")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def custom_exc_handler(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()
    return []

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), custom_exc_handler)

try:
    from google.colab import userdata
    for k in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "hf_token"]:
        try:
            tok = userdata.get(k)
            if tok:
                os.environ["HF_TOKEN"] = tok
                os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
                print(f"[Secrets] Loaded Hugging Face token from Colab Secrets ({k}).")
                break
        except Exception:
            pass
except ImportError:
    pass

print("[Ready] Credit guard active. VM will self-terminate on error or completion.")

In [ ]:
# Step 2: GPU check + clone repository
import torch
assert torch.cuda.is_available(), "Switch Colab runtime to GPU (L4 24GB): Runtime -> Change runtime type -> GPU (L4)"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB | torch {torch.__version__} | CPUs {os.cpu_count()}")

repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/vla-reachability"
target_dir = "/content/vla-reachability"
if not os.path.exists(target_dir):
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
os.chdir(target_dir)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Step 3: Mount Drive first (HF cache + results live there, so checkpoints download once)
from google.colab import drive
drive.mount("/content/drive")
os.environ["VR_RESULTS"] = "/content/drive/MyDrive/moonshot_results/vla-reachability"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.makedirs(os.environ["VR_RESULTS"], exist_ok=True)
print("Results dir:", os.environ["VR_RESULTS"], "| existing:", sorted(os.listdir(os.environ["VR_RESULTS"])))

In [ ]:
# Step 4: Installs (LeRobot 0.7 with LIBERO = hf-libero/robosuite 1.4/mujoco, SmolVLA and pi0.5 deps)
subprocess.run(["apt-get", "-qq", "install", "-y", "libegl1", "libgl1", "libglew-dev"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake"], check=False)  # hf-egl-probe builds with CMake (no py3.13 wheel)
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], capture_output=True, text=True)
if _r.returncode:  # Colab hides child-process output: print pip's real error before failing
    print(_r.stdout[-4000:]); print(_r.stderr[-6000:])
_r.check_returncode()
if os.environ.get("HF_TOKEN"):
    import huggingface_hub
    huggingface_hub.login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
# fresh interpreter: builds one LIBERO env (downloads assets once) to check EGL rendering
print(subprocess.run([sys.executable, "-c", "import sim, lerobot; e = sim.ReachEnv(sim.suite_obj(), 1, sim.SUITE, obs_type='pixels_agent_pos').prepare(); "
                      "print('lerobot', lerobot.__version__, '| objects', e.names, '| distractor', e.distractor.tolist())"],
                     capture_output=True, text=True).stdout[-2000:])

In [ ]:
# Step 5: Pre-flight on this machine (exact cross-env restore, predicates on a real scene, worker pool, SmolVLA forward on GPU).
# Nothing may be skipped (VR_REQUIRE_SIM=1, VR_REQUIRE_GPU=1). ABORTS (and unassigns) on failure.
env = dict(os.environ, VR_REQUIRE_SIM="1", VR_REQUIRE_GPU="1")
if subprocess.run([sys.executable, "-m", "pytest", "-v", "-p", "no:cacheprovider", "test_reach.py"], env=env).returncode:
    raise RuntimeError("pytest pre-flight failed -> campaign aborted (INVALID)")
print("[Pre-flight] passed")

In [ ]:
# Step 6: Pilot + full campaign (resumable). Stops after the pilot on INVALID / abandon signal / budget overrun (see CAMPAIGN.md).
p = subprocess.Popen([sys.executable, "-u", "run_all.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "WARN" not in line and "Fetching" not in line:
        print(line, end="")
if p.wait():
    raise RuntimeError(f"run_all.py stopped (exit {p.returncode}); read the message above; re-run the notebook to resume from Drive")

In [ ]:
# Step 7: Results
print(open(os.path.join(os.environ["VR_RESULTS"], "RESULTS.md")).read())

In [ ]:
# Step 8: Flush Drive & unassign GPU
auto_teardown()